# Run 단위 일관성 reset

리팩토링 전후의 데이터가 섞이지 않도록, 하나의 `run_uid`에 귀속된 **DB 행과 로컬 실험 산출물 전체**를 같은 미리보기 계획으로 정리합니다.

기본 모드인 `complete_run_reset`의 대상 범위는 다음과 같습니다.

- 임베딩 추출 및 임베딩 벡터 전처리 산출물
- PCA/PQ 학습 모델·변환 결과·codebook
- Grad-CAM, leave-one-out(LOO) template 및 saliency 특징
- 평가 표·그림·로그
- manifest가 정확히 지목한 run result bundle과 pointer
- 해당 `run_uid`에 귀속된 PostgreSQL embedding/template/평가 결과와 `research_runs` 레코드

안전 원칙:

- 기본값은 DB 연결과 실행이 모두 꺼져 있습니다.
- 미리보기의 행 수·파일 수·바이트 수·보존 대상·경고·차단 사유를 먼저 확인합니다.
- 동일 계획에서 출력된 확인 문자열이 없거나 계획이 바뀌면 reset하지 않습니다.
- 로컬 대상은 영구 삭제하지 않고 `runs/database_cleanup/quarantine/`으로 격리합니다.
- DB 행은 커밋 후 감사 JSON만으로 복구할 수 없습니다. 감사 기록은 삭제된 데이터를 백업하지 않습니다.
- 승격된 논문 결과, 완료 run, lineage를 검증할 수 없는 산출물은 각각 별도 보호 해제가 필요합니다.


In [14]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import display
from sqlalchemy.orm import Session


def find_project_root(start: Path) -> Path:
    resolved = start.resolve()
    for candidate in (resolved, *resolved.parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("C:/ronbun 프로젝트 루트를 찾지 못했습니다.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.database.cleanup import (
    SCOPE_EXACT_RUN_UID,
    SCOPE_LEGACY_NULL_RUN_UID,
    build_cleanup_plan,
    collect_run_inventory,
    collect_table_totals,
    execute_cleanup_plan,
    write_cleanup_audit,
)
from research.database.connection import (
    check_database_health,
    create_database_engine,
    ensure_database_schema,
)
from research.database.reset import (
    build_run_reset_plan,
    execute_run_reset_plan,
)
from research.database.settings import load_database_settings

print(f"PROJECT_ROOT={PROJECT_ROOT}")


PROJECT_ROOT=C:\ronbun


## 1. 실행 설정

`RESET_MODE="complete_run_reset"`이 권장값입니다. 이 모드는 exact `run_uid`의 DB 및 로컬 closure를 함께 계획합니다.

`advanced_database_cleanup`은 특정 허용 테이블만 정리하거나 과거 `run_uid IS NULL` embedding 행을 정리할 때만 사용합니다. 이 호환 모드는 로컬 산출물을 다루지 않으며 `research_runs` 부모 레코드 삭제 옵션도 노출하지 않습니다.


In [15]:
# 권장: DB와 run 전용 로컬 산출물을 한 계획으로 정리합니다.
RESET_MODE = "complete_run_reset"  # 또는 "advanced_database_cleanup"

# 1단계 미리보기 시에만 True로 바꿉니다.
CONNECT_TO_DATABASE = True
APPLY_ADDITIVE_SCHEMA = True  # 새 모델의 누락 테이블만 생성할 때 한 번만 True
RUN_UID = "20260727-R001-c305da87"  # complete_run_reset에서는 정확한 run_uid가 필수입니다.

# 보호된 상태는 각각의 의미를 확인한 뒤 필요한 항목만 True로 바꿉니다.
ALLOW_COMPLETED_RUN_RESET = False
ALLOW_PROMOTED_RESULTS_RESET = False
ALLOW_UNVERIFIED_LINEAGE_RESET = False

# 2단계 실행: 미리보기 확인 문자열을 붙여 넣은 뒤에만 True로 바꿉니다.
EXECUTE_RESET = True
CONFIRMATION_TOKEN = "RESET 20260727-R001-c305da87 DB_ROWS=13195 FILES=18 BYTES=5518908 d274c560226d"
WRITE_AUDIT_OUTPUT = True

# advanced_database_cleanup 전용 설정입니다.
ADVANCED_SCOPE_KIND = SCOPE_EXACT_RUN_UID  # 또는 SCOPE_LEGACY_NULL_RUN_UID
# legacy null 범위에서는 ["image_embeddings"]만 사용해야 합니다.
ADVANCED_TABLE_GROUPS_SELECTED = ["all_run_scoped_data"]
ADVANCED_TABLE_NAMES_SELECTED = []

{
    "reset_mode": RESET_MODE,
    "connect": CONNECT_TO_DATABASE,
    "apply_additive_schema": APPLY_ADDITIVE_SCHEMA,
    "run_uid": RUN_UID or None,
    "allow_completed_run_reset": ALLOW_COMPLETED_RUN_RESET,
    "allow_promoted_results_reset": ALLOW_PROMOTED_RESULTS_RESET,
    "allow_unverified_lineage_reset": ALLOW_UNVERIFIED_LINEAGE_RESET,
    "execute_reset": EXECUTE_RESET,
    "write_audit_output": WRITE_AUDIT_OUTPUT,
    "advanced_scope_kind": ADVANCED_SCOPE_KIND,
    "advanced_table_groups": ADVANCED_TABLE_GROUPS_SELECTED,
    "advanced_table_names": ADVANCED_TABLE_NAMES_SELECTED,
}


{'reset_mode': 'complete_run_reset',
 'connect': True,
 'apply_additive_schema': True,
 'run_uid': '20260727-R001-c305da87',
 'allow_completed_run_reset': False,
 'allow_promoted_results_reset': False,
 'allow_unverified_lineage_reset': False,
 'execute_reset': True,
 'write_audit_output': True,
 'advanced_scope_kind': 'exact_run_uid',
 'advanced_table_groups': ['all_run_scoped_data'],
 'advanced_table_names': []}

## 보존 경계

완전 reset에서도 다음 공유 자원은 삭제하거나 격리하지 않습니다.

- 원본 데이터셋 `data/raw/**`
- 여러 run이 공유하는 `data/interim/**` 및 공용 정렬 얼굴 crop
- 사전학습 checkpoint와 모델 registry
- 공유 PostgreSQL `images` 테이블
- 다른 `run_uid`의 DB 행과 로컬 산출물

`results/paper/**`처럼 논문 결과로 승격된 산출물은 기본적으로 차단합니다. 정말 폐기할 때만 `ALLOW_PROMOTED_RESULTS_RESET=True`를 사용합니다. 경로 또는 manifest만으로 귀속 run을 증명할 수 없는 파일도 `ALLOW_UNVERIFIED_LINEAGE_RESET=True` 없이는 대상이 되지 않습니다.


## 2. DB 연결 및 읽기 전용 인벤토리

이 단계는 스키마와 데이터를 변경하지 않습니다. 접속 비밀번호는 화면에 출력하지 않습니다.


In [16]:
engine = None
database_health = None

if CONNECT_TO_DATABASE:
    database_settings = load_database_settings()
    display(pd.DataFrame([database_settings.redacted()]))
    engine = create_database_engine(database_settings)
    database_health = check_database_health(engine)
    if database_health["schema_issues"]:
        raise RuntimeError(
            "DB schema column/constraint가 현재 모델과 다릅니다. "
            f"reset하지 말고 확인하세요: {database_health['schema_issues']}"
        )
    if database_health["missing_tables"]:
        if not APPLY_ADDITIVE_SCHEMA:
            raise RuntimeError(
                "DB에 새 모델 테이블이 없습니다. 첫 번째 미리보기 실행에서만 "
                "APPLY_ADDITIVE_SCHEMA=True로 설정하세요. "
                f"missing={database_health['missing_tables']}"
            )
        ensure_database_schema(engine)
        database_health = check_database_health(engine)
        if database_health["missing_tables"] or database_health["schema_issues"]:
            raise RuntimeError(
                "additive schema 적용 후에도 DB health가 정상화되지 않았습니다. "
                f"missing={database_health['missing_tables']}, "
                f"issues={database_health['schema_issues']}"
            )
    display(
        pd.DataFrame(
            [
                {
                    "database": database_health["database"],
                    "user": database_health["user"],
                    "vector_extension_version": database_health[
                        "vector_extension_version"
                    ],
                    "missing_tables": len(database_health["missing_tables"]),
                    "schema_issues": len(database_health["schema_issues"]),
                }
            ]
        )
    )
else:
    print("안전 기본값: DB에 연결하지 않았습니다. CONNECT_TO_DATABASE=True로 바꾸세요.")


,driver,host,port,database,user,password,password_source,connect_timeout_seconds,pool_pre_ping,echo_sql
0,postgresql+psycopg2,localhost,5432,postgres,postgres,***,local_config,10,True,False


,database,user,vector_extension_version,missing_tables,schema_issues
0,postgres,postgres,0.8.4,0,0


In [17]:
table_totals = []
run_inventory = []

if engine is not None:
    with Session(engine) as session:
        table_totals = [item.as_dict() for item in collect_table_totals(session)]
        run_inventory = [item.as_dict() for item in collect_run_inventory(session)]

    print("전체 관리 테이블 행 수")
    display(pd.DataFrame(table_totals))
    print("run_uid별 행 수")
    if run_inventory:
        display(
            pd.DataFrame(run_inventory).sort_values(
                ["run_uid", "table_name"], na_position="first"
            )
        )
    else:
        print("run_uid가 연결된 행이 없습니다.")
else:
    print("DB 연결이 꺼져 있어 인벤토리를 건너뜁니다.")


전체 관리 테이블 행 수


,table_name,row_count,exists,cleanup_policy
0,images,13195,True,preserved_shared_source
1,research_runs,0,True,exact_run_uid_only_with_fk_children
2,template_embedding_128,0,True,guarded_run_scope
3,template_embedding_256,0,True,guarded_run_scope
4,template_embedding_32,0,True,guarded_run_scope
5,template_embedding_384,0,True,guarded_run_scope
6,template_embedding_448,0,True,guarded_run_scope
7,template_embedding_512,0,True,guarded_run_scope
8,template_embedding_64,0,True,guarded_run_scope
9,embedding_128,0,True,guarded_run_scope


run_uid별 행 수


,table_name,run_uid,row_count,research_run_status
0,embedding_512,20260727-R001-c305da87,13195,None


## 3. reset 계획 미리보기

여기까지는 읽기 전용입니다. `complete_run_reset`은 DB 행과 로컬 대상을 하나의 digest로 묶습니다. DB 표, 로컬 표, 요약, 보존 자원, 경고와 차단 사유를 모두 확인한 뒤 확인 문자열 전체를 `CONFIRMATION_TOKEN`에 복사합니다.


In [18]:
run_reset_plan = None
advanced_cleanup_plan = None

if RESET_MODE not in {"complete_run_reset", "advanced_database_cleanup"}:
    raise ValueError(f"지원하지 않는 RESET_MODE: {RESET_MODE}")

if engine is None:
    print("DB 연결이 꺼져 있어 미리보기를 건너뜁니다.")
elif RESET_MODE == "complete_run_reset":
    if not RUN_UID.strip():
        print("complete_run_reset 미리보기를 만들려면 RUN_UID를 정확히 입력하세요.")
    else:
        with Session(engine) as session:
            run_reset_plan = build_run_reset_plan(
                session,
                run_uid=RUN_UID,
                project_root=PROJECT_ROOT,
                allow_completed_run=ALLOW_COMPLETED_RUN_RESET,
                allow_promoted_results=ALLOW_PROMOTED_RESULTS_RESET,
                allow_unverified_lineage=ALLOW_UNVERIFIED_LINEAGE_RESET,
            )

        print("DB reset 대상")
        display(
            pd.DataFrame(
                [item.as_dict() for item in run_reset_plan.database_plan.table_rows]
            )
        )
        print("로컬 quarantine 대상")
        display(pd.DataFrame([item.as_dict() for item in run_reset_plan.local_targets]))
        display(
            pd.DataFrame(
                [
                    {
                        "run_uid": RUN_UID,
                        "total_database_rows": run_reset_plan.total_database_rows,
                        "total_files": run_reset_plan.total_files,
                        "total_bytes": run_reset_plan.total_bytes,
                        "plan_digest": run_reset_plan.plan_digest,
                        "executable": run_reset_plan.executable,
                    }
                ]
            )
        )
        print("전체 계획(JSON 호환)")
        display(run_reset_plan.as_dict())
        print("보존 자원")
        for resource in run_reset_plan.preserved_resources:
            print(f"PRESERVED: {resource}")
        for warning in run_reset_plan.warnings:
            print(f"WARNING: {warning}")
        for blocker in run_reset_plan.blockers:
            print(f"BLOCKED: {blocker}")
        if run_reset_plan.confirmation_token:
            print("확인 문자열(공백 포함 그대로 복사):")
            print(run_reset_plan.confirmation_token)
else:
    advanced_scope_is_configured = (
        ADVANCED_SCOPE_KIND == SCOPE_LEGACY_NULL_RUN_UID
        or (
            ADVANCED_SCOPE_KIND == SCOPE_EXACT_RUN_UID
            and bool(RUN_UID.strip())
        )
    )
    if not advanced_scope_is_configured:
        print("exact_run_uid 고급 정리에는 RUN_UID가 필요합니다.")
    else:
        with Session(engine) as session:
            advanced_cleanup_plan = build_cleanup_plan(
                session,
                scope_kind=ADVANCED_SCOPE_KIND,
                run_uid=RUN_UID or None,
                table_groups=ADVANCED_TABLE_GROUPS_SELECTED,
                table_names=ADVANCED_TABLE_NAMES_SELECTED,
                allow_completed_run=ALLOW_COMPLETED_RUN_RESET,
                project_root=PROJECT_ROOT,
            )
        display(
            pd.DataFrame(
                [item.as_dict() for item in advanced_cleanup_plan.table_rows]
            )
        )
        display(pd.DataFrame([advanced_cleanup_plan.as_dict()]))
        for warning in advanced_cleanup_plan.warnings:
            print(f"WARNING: {warning}")
        for blocker in advanced_cleanup_plan.blockers:
            print(f"BLOCKED: {blocker}")
        if advanced_cleanup_plan.confirmation_token:
            print("확인 문자열(공백 포함 그대로 복사):")
            print(advanced_cleanup_plan.confirmation_token)


DB reset 대상


,table_name,row_count
0,research_calibration_results,0
1,research_search_results,0
2,research_templates,0
3,research_splits,0
4,template_embedding_512,0
5,template_embedding_448,0
6,template_embedding_384,0
7,template_embedding_256,0
8,template_embedding_128,0
9,template_embedding_64,0


로컬 quarantine 대상


,kind,relative_path,file_count,byte_count,state_digest,owner_manifest,owner_manifest_sha256,promoted
0,run_workspace,runs/lfw/2026/07/27/20260727-R001-c305da87_the...,18,5518908,6e9f8df9c0cdb5c7aea150a6725f51e43bc04060320aa7...,runs/lfw/2026/07/27/20260727-R001-c305da87_the...,9197f40a4e15a677ecd362b8bde22e11d61e0631fce5df...,False


,run_uid,total_database_rows,total_files,total_bytes,plan_digest,executable
0,20260727-R001-c305da87,13195,18,5518908,d274c560226d3e9533fda7aab2ff2d108121b6883bc698...,True


전체 계획(JSON 호환)


{'reset_plan_version': 1,
 'reset_kind': 'complete_run_reset',
 'run_uid': '20260727-R001-c305da87',
 'project_root': 'C:\\ronbun',
 'database_plan': {'plan_version': 2,
  'database': 'postgres',
  'database_user': 'postgres',
  'scope_kind': 'exact_run_uid',
  'run_uid': '20260727-R001-c305da87',
  'selected_tables': ['research_calibration_results',
   'research_search_results',
   'research_templates',
   'research_splits',
   'template_embedding_512',
   'template_embedding_448',
   'template_embedding_384',
   'template_embedding_256',
   'template_embedding_128',
   'template_embedding_64',
   'template_embedding_32',
   'embedding_512',
   'embedding_448',
   'embedding_384',
   'embedding_256',
   'embedding_128',
   'embedding_64',
   'embedding_32',
   'embedding_pq',
   'research_runs'],
  'table_rows': [{'table_name': 'research_calibration_results',
    'row_count': 0},
   {'table_name': 'research_search_results', 'row_count': 0},
   {'table_name': 'research_templates', 'row

보존 자원
PRESERVED: PostgreSQL images table
PRESERVED: data/raw
PRESERVED: shared data/interim dataset manifests and aligned crops
PRESERVED: pretrained checkpoints and model registries
PRESERVED: other run_uid lineages
PRESERVED: runs/database_cleanup audits and quarantine payloads
PRESERVED: local paths without an exact owner manifest
확인 문자열(공백 포함 그대로 복사):
RESET 20260727-R001-c305da87 DB_ROWS=13195 FILES=18 BYTES=5518908 d274c560226d


## 4. 확인된 계획 실행

`EXECUTE_RESET=True`만으로는 실행되지 않습니다. 같은 실행에서 만든 계획과 `CONFIRMATION_TOKEN`이 정확히 일치해야 합니다. `complete_run_reset`은 로컬 대상을 먼저 격리하고 DB 트랜잭션이 실패하면 격리를 되돌립니다. 성공 후 로컬 격리본은 복구 가능하지만, 커밋된 DB 행은 별도 백업 없이는 복구할 수 없습니다.


In [19]:
reset_report = None
audit_path = None

if EXECUTE_RESET:
    if engine is None:
        raise RuntimeError("EXECUTE_RESET=True이지만 DB 연결이 없습니다.")
    if not CONFIRMATION_TOKEN:
        raise RuntimeError("미리보기의 CONFIRMATION_TOKEN을 정확히 입력하세요.")

    if RESET_MODE == "complete_run_reset":
        if run_reset_plan is None:
            raise RuntimeError("실행 가능한 최신 complete run reset 계획이 없습니다.")
        if not WRITE_AUDIT_OUTPUT:
            raise RuntimeError("complete_run_reset은 감사 기록을 끌 수 없습니다.")
        reset_report = execute_run_reset_plan(
            engine,
            run_reset_plan,
            confirmation_token=CONFIRMATION_TOKEN,
            project_root=PROJECT_ROOT,
        )
        display(pd.DataFrame([reset_report.as_dict()]))
    else:
        if advanced_cleanup_plan is None:
            raise RuntimeError("실행 가능한 최신 advanced DB 정리 계획이 없습니다.")
        reset_report = execute_cleanup_plan(
            engine,
            advanced_cleanup_plan,
            confirmation_token=CONFIRMATION_TOKEN,
            project_root=PROJECT_ROOT,
        )
        display(pd.DataFrame([reset_report.as_dict()]))
        if WRITE_AUDIT_OUTPUT:
            audit_path = write_cleanup_audit(
                reset_report,
                PROJECT_ROOT / "runs" / "database_cleanup",
            )
            print(f"감사 기록: {audit_path}")
else:
    print("안전 기본값: EXECUTE_RESET=False이므로 아무것도 변경하지 않았습니다.")


,completed_at_utc,run_uid,plan_digest,database_report,quarantined_targets,quarantine_dir,audit_path,warnings
0,2026-07-26T16:16:15.189537+00:00,20260727-R001-c305da87,d274c560226d3e9533fda7aab2ff2d108121b6883bc698...,{'committed_at_utc': '2026-07-26T16:16:15.1895...,"[{'kind': 'run_workspace', 'source_relative_pa...",runs/database_cleanup/quarantine/20260726T1616...,runs/database_cleanup/quarantine/20260726T1616...,[]


In [20]:
if engine is not None:
    engine.dispose()


## 권장 실행 순서

1. `RESET_MODE="complete_run_reset"`, `CONNECT_TO_DATABASE=True`, `EXECUTE_RESET=False`로 커널을 재시작하고 위에서 아래까지 실행합니다.
2. DB 행, 로컬 격리 대상, 파일 수·용량, 보존 자원, 경고와 차단 사유를 확인합니다.
3. 완료 run·논문 승격 결과·검증 불가능 lineage가 차단되면, 폐기 의도가 분명한 보호 항목만 해제합니다.
4. 다시 미리보기를 만들고 출력된 확인 문자열을 `CONFIRMATION_TOKEN`에 그대로 붙입니다.
5. `EXECUTE_RESET=True`로 바꾸고 커널을 재시작한 뒤 위에서 아래까지 다시 실행합니다.
6. 실행 보고서와 `runs/database_cleanup/`의 감사 JSON, quarantine 경로, DB 재조회 결과를 확인합니다.

개별 embedding 테이블 또는 legacy null 행만 정리할 때에만 `advanced_database_cleanup`을 사용합니다. 이 모드는 로컬 전처리·결과 산출물을 정리하지 않으므로 완전한 run 재실행 준비 용도로 사용하면 안 됩니다.
